# MCAP → MP4 (folder Drive, satu per satu)
Jalankan sel **berurutan**. Berhenti dan cek hasil di tiap langkah sebelum lanjut.

Cara kerja: tiap file `.mcap` disalin ke disk lokal Colab → diproses di proses Python terpisah (RAM bebas total setelah selesai) → MP4 disalin ke Drive → file lokal dihapus.

## Langkah 1 — Hubungkan Drive & siapkan skrip
Skrip ekstraksi sudah tertanam di notebook ini, tidak perlu upload apa pun. Mendukung dua jenis file: `ego_*.mcap` (dua kamera, protobuf) dan file ROS2 bertopik `/head/camera/image_compressed` (satu kamera, H.265). Jenisnya dikenali otomatis. Chunk `zstd` dan `lz4` juga didukung.

In [ ]:
import base64, zlib, hashlib, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# dukungan chunk MCAP terkompresi (zstd / lz4)
for pkg in ('zstandard', 'lz4'):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], capture_output=True, text=True)
    if r.returncode != 0:
        print(f'PERINGATAN: gagal install {pkg}:', r.stderr.strip()[-200:])

_FILES = {
    'mcap_to_mp4.py': ('eNrNPGtz3EZy3/krcLi6M2CulrskRdMbrRJHscquilNKyfGHUCwUFpglccICCIBdkqJZpXP5ITm2pZzv7DtLVy7n/LjyxY+7uGxZ8tn/5aIlqU/+C+nueWDw2KXOlQ+hTC4w09PT3dPTr5n1MI1HhuMMx/k4ZY5jBKMkTnPDjaI4d/MgjrKFBdmWbiVumjH5/rMsjuRzlqdjL5dveTBiC0PE7Lu564VulrFMolZNLWMYsNDngImbb4fBQAJdgNeFhYVzTzxz0egb++YS24qXPHfEUnepY/YMkz87HbNllDq7WmfXPAAcPhsag+6aNbF7RsqAzUhQ205c77Jlnn0KcExsDre8qsNN2nnsDPZyllkrMM8g2DIF3MrybHxPa/jiXStvGUkBiiNDFlmJbSwa6/gnh9+EoIcCfAKSCcujeAcnZWOyieOQ1mGITwmfbAxMDlpGXKNrHCFlDsrZMs8guwS20dmEDxi/zIcDZQ8x/Ona8FU+fG31YYb/c234ulijEcALDAsG/ETY3S/IokYpj424h2OjzbbPvNhnljnOh6fWURtSloSux0ybo48kei68hgkk4c0TlLFkl4PEGbnJwxKqj524aRDl+sgJDOvQU6aedraDkBnPpmPGYfDHg16gaFM1ANq+0VWvE+P5vmF5xk+Nzu5j523jzBkjU53BEMafgZ71ToFRIxG0LVbtGSJ+TBDsxaMkZbB3fWcS+Cx2EncvjF3fGmVbgoNY0Y3bGl7+KY4YvQ9Huf7K2YqBENT+AgH+XGZ7XIRCQtCr5Eio0ExAN8AZZ88aK6pjJxetPwWaNX6xHegqs+vMnYPEHEd5EI1ZHVW3jGrihoAJUAgtWd8sddPqrKsmFio8y2U80YkkVWeKGmaKmmY6PZ/i1QY8qxqejFV0xQ0yZjznhmP2ZJrGqTU0x1E2TtBYM99I0jiPB2OYPEiZke8lzNjfyQ9Muy7KZfAtvlzSvrFSnkdoEZDbKL7y2NXyWK5xMHSOTWiH8Q5LrdIexTlbOFqofZCz1EkBRepnFvqlFvK3hTuhj/osNDcHBxly7wDTIlw7A5dp2fDhZMEVrkZAuxzcU09Wp6UP59TsBPk2RxMnsEXMdGDahpsZw54uxGE7ZbADwXP8CGyCeWl3/fFnzj1xoXMpvRSZJ67ZPuI/MILMAPcOwjRwLAg0ZNpSNVmg2awM2zkLQ8tu4EhpVwICEoR37SpOpCROylMVq1PhaMcB81FgW69hQ+siwEhE6w+HmHkF0pLfEi5LogSvVZsyTqAV5+rsdtbq09WsirSb62g5xO9qxRCNEm4ayCkCdTXDEEQRSwUM92yNUNyI42Yxr2S5b9bJEwEX9MLeclO/DoAzwTQKov2vgOkfmPQOoFg27Tj+atUQaOSO3F0nHufJmO+Q/swQgZjprtXlTdZA8RReWZ3NEnS2hymEgjNZUhA6/dQ5Y9IGZZpvGAsXanjb4+hyz9jHJt0yEkGFHy3vwkB6TE5VA6+0uah3I4g3G/rLoYJq5/to5gKIBQsoUJuBdb3WvkemGWhqKZJ6CEvTzUKDffWuucbGOKUJBdHHFfMjorUkDHIHchm26wwstPNCgj82LsKqg5Zl4zBHI+tSbH1qsHcKP43Mc8E3D8aQ5IxHSWYMWL7DGIa0LigWOpeM22uarj0MIh82AY/kEAQ9wgbnNwxGAfolpJcoANp5BBNUYr4Alxqhi1XGtcVBfAYw9x3418XlKXnWBOO7sm4MwJRdLsOchRnQfyLCjQTI6G42hEmQliVcNwikNTcs0IBbWlwG02XG2X6VmzpdXFhtNwGP51uWQGdrzKGOYoojdQSsRyHbYZwagb/bMixCJEaD6hksGmMKmDPek2k7J8pI73EhNbzUw2RPtgFoobu7iaYd2BGvYjMKnCSQYmELPGANxMJtRFkvYpvtFDZakNAKloMihC1LCDiUAsHOcjoxlkFKhEEWB1AJF70iwRAkd23KBlbOC/h0kCUCnKtpih6PBiz3NjXRoka7aeruiSDpCkvjIjtBiXsoYBhfiks42FkZo3kNsV0Z00z3qPHvFaKSg/knLgXlNai/fBk6pfSNPCJgQktAxQbj74M84/SgOBwniMA0OFbGwmHL0EwDCQca275YwnIr37QKDw8YCEekIZiU2EShOSi01I22YNEqZhyBrQkmbbBozxuWxaffEPMtLUF+gUtqPQYbUjT+xFi3cYUr4ZToLVl8meYVRA+CnObQUnUcJ+MzBTdmAmxh9hJy06WPb7ApYtEaqLK6yDf1o2HswsoWuHgzrODfqZqRWMyno2HcE1Gzn2/3QLiou93Hlzll2yzY2s5Vc2edN4MfwVjXycDbMdnLu/KApeUWCRz4nkK0IKMzVyHv7K51+I/ojGCru9AHfksmB8Jwi38m9wtswkKFWNDtbYMHdsvTheOR6/gsKdjUYZt6coYBEDjD0N1jaVZGpzojlkGEUnRyOyG5Rnk4RKM1EIFbNh4IVSjJEflL+YIt20qWWqvQUU2eWudpW5Op1r6ybFflqfWurttFaW5NleaUXDVQkSIkieaSk7IL0fZmmU8+UFojwIg7R/NO4Yw+ME0CUW+WEVgHba9Pxj1kSZjk4+YRiL492Qg2Z2AhW7lonIbfFXxZXS/5n3D20PWS6ymteYvWuKWvaUusYUtfshZfDqlbWD0GBJkFv4KDAWoK2maLPBR22As6Fav8VfCtawjv4GrcoG6ZIrJGXUtpyQnqLikBS2hrm47PphrRE4n2ktOrUESmqjKUG6o6PqlPBbKQDYH6FMEx1oV4axDneTwqxrZmPxQ+bAf3ukYxKBZWI1ZEMLNcQCKpy2XeuhyosOKcpVN9xPuoYSGNoGJEZDGn4BGhthEKiMfyNVEv9iwZsYoUvKZGoYxo/y2avCXQtx5mwVuCl5aYsSUmUQuOm6UlVKrR8UDaACFjT1U8hY1fEGVMeI3jsGngsykkWHxc6u70IDTO8o18DMg26JHQbLZo/OYmFiMwlbJg27iQojhD18vjdK+PsEK/XUhcwbb7gZdvwF5rGRqeOQjc0cAHsvdXlntgA2HlV8TnKn4eCA1E/0pSnokIO4WFCiC65X7keSr4yrov3/XjKPg3y8UjkZ7U7gkVfUD13EK/XWlKJzJ7c31fVZxzFB+sLH+lah3tgsArsGL9j+qD8PmjvmFuL6+d1qoDTeUwwkD1sH0YddAiumicMC4ATelCKZEUVAhnB3E4guyLiNwmix0hdzT6QFnxoq0gCre+HKibZZIOMd0WC10y0SRTvXsj38QItFr8oarpygqJBfOU0pCVlU27obhNZBFcwNe/MNtiAqy49wHnntVdM870YRb4s9wlJjnhKBNlyzgy0hKUNGkIEoRo8NOFjL7CS4EKAgHUUdRPW7OGOs6+xhrsLbEs8lUllyT5Fs5q2+qAIwyZBwsLFMwu9HrbuPAhLfJBgZzeSYF6fHdbfOmpCUnHw8ti9dFiY5kROuYVmOVD2cHHVO7q7HYqJe962cgLfHEc1V1rLAk687sjd8ROKD06J/SrM7KmTinMDSB0kzJQrT5HdTbF6+n/M15Xlud2r63+kG6+zn3FUXuL5RaQVN+EBCgsHledh6wS6+YOZ5p1HgfkbcS9emVa9Mt6P+p0feaamYUNiBRvNpvbOacaDWcgraaDkZIr51OKDTkMYJcChLT3FQuPTKiNPdeyR7FBIjIyctgZbMExugbYiMLom7aO9YfYoIZpR0GWBdGW8dSTz50znrtwceki/F64cLFhYklYKRMpbFRh/4HVDd3cAVinp4Xs4sisWreZ53XUMEjJ5YWEyLbprFrtYk6fNJ888LF4xkXlEmlKNeCNzmabOwc8PdJXWQCIZd6eeJ6F3qUlwhjQMJURJA3MqDyrW3knLO1yPnrmjLFGhRTqo2QUmk5jUwkaokSBTcqBunncSFVl/qqymfLU1EthpY4FLFFn93yn09Eb+bUNaD/XMvCvIk6E1j+F+Jt61lVPkfXj4Xa1V8/9qX+zQgOKsyIpIY5ybYAGo3hWeOmpDCNyKywzIcwywqxU8K4UaWpts2hZtATHawiARUsPFM10LUbuPrtBi1VnWZNLCHQ11i7PDEzQuXOmvEKTqBBgCLGKTLNK8NiB8GaQxSMTExZQkM7uMqwsvlAz/K6NktXuNpt0TYlxNNkGQwoq7Y/lgc3IzdNg1+gLHF2sFtkS5ZyHvwJyVRShSrZ1yFlBioCVDv3XjACr2fwJqW6Yn/S6K5m/1DEhk+t24EXwVrQur4rhy0rI+WUQCSBuGTuQr/3/EArSxIXy2CyhdGstUjiC1XUpGXv+QyEjwLFDld41iXJbvKozqzzzZ9pGFuXpnigoIgFrYpZuaVlkK21mypJtvYUnzJqU1ksCrb7bTXNQKQlbl1cL9TgPP/g2w7w3LAIyW9JMPgnfsLipWpxlJZ1RHE8s4UxaRm0OCNKGw4yp855RPAmYQ6XDrqzQjpgfuLztcVW25YCwvNCaYqhAxkRMZAPDCtNSwYvPQnK9CuMSdWqzcIQVVDRMnBUSGUoYedYgDOGfJQKliojFVgkZLzOb7Z/FQWSRL8NaBs9IgpaRVQ7FJDV4UocuW7jxPCvRk5XpQTJwKhsZpXWnNznUKw31mllpelAYrpQwXGm2WTVh6FxTZ5viE847Ma7AxURerE/kxTUupRbV8Mcca4xYFZDAOqCys0A6CAHpjL2Mhxy48otc4ItceItcAotEn1CwFFReUYpvdaGKznEaGryzi07LFFSNYHZFFb4AjBgxGUkDiCNOsmLFvvQ5FqSGczIQAdDI3/YLavHtBKcj9035XdpYMimnT59b1YjgNVM/TIt58K1hHhPDf1O3imSxzOcwK3jKjfyQpeosFuK7y0pK+AIIyWcps8BtTVuUHKtmVDAcuDCOpLDIiVwk+dt2PbpAO4bAEzUJRgxqOroXDHRIs7eTBjlzINTAM82WMc8ANlUwtplLK0OxjjBP3HKdZE7xNPnRkgGzm45NlrVQz1M2D6dFTujGJ8OrD4vanRGxfYoESOLVt2ylBMCHqOPhceWeDk69KOxtu3wm/7AMy01dXMnD82h+I2+nfiNv2Kal4azWWonpWisqKUYO2Xik26sGc0UaO4Le8p2B5guF9UVSx0eRz3al1jQ6ArABlVMoSSwHqAhzNh1iqjItXIe3WO4M8ewnHqceaHkAm4rtJmHgBXmR4suWXvWkWHbo6lugArdsZkBAEEdt/F6Cqc7hALLNdgOwrfppikCK124RvI1ljozWi45snJzt5haLvBis3VZfXCS1barxmMAF7N2VTnljr3RkBMewqMh8h5NX5hgzT9I8P4AurXiN7VQOFV3VOsPFPcjFRk/uBkDAv8A7A92EzPbUKT4yxkcxlJexB3G+3S6qHASm3YJxR7z4gA/8IsqIyduxbdhdGSq/ZbZHnpuY4nRoaO4j2AFvFBWLEakUoS/2sWAdJygtEoHr44gOPo6qfzXuNUybda2mcmp1CPEXpyMqJfl0PQ5Aq9pLjYpjfqWH88y2YsdsuGtXx7mx2tus32eVULPFeQLu4mWjd+p0eQYhD2kDS9IdEun7qNUFCrvXWfXFktkL9XtchYAzurOooWxvhfHAMh8Vg2VIwWtcuLR5KqCF9aJnMmECqSiv8Ve1DzfV7uS45un6M2I6gRv1zIJ9cylC527CBw/OBKbylhRU8G3pB5kXT1iqtmUax9Lw4CNF/nEO+z+Lw0nlwFGIJiE2E3J8BCsEVLnOWVjIdpA5RLJNeztpZ+PhMFBqR5dYuXQbEPAR4J4gB8ITLaISG+kBCyIFsQJMXX1koZtkuJjMi0H7ZCAArGd4DkHHJn4wGcU+XilA/4FaI6FtMG9rMm+GGGaco1WB3qwYhkhaxloluTb3aY5eZ9k/6O2LoeINx9OjqW5lXZAuZNbNrNAd4LktL+/xS378oie9VO9sCWh61odQgzYOA5S5iKtos9KXWVQH1gpDxu8fnHdhWy1ofHhuGBY3zGCsmEG/ixjjeTx+Xa49imGjxFHg6fVbEEW2TdhppHHW6PArnHiGdla0lq5dlqkCZbUKDrRjAMKCL2oK3rBzqgDH+2i9ymV5XRwAPVsaEm9R5mOw6eiqESiFBdk4l8kSsfAoVZborATZ1C708ZtDqJtD0zD2i1U++P6bd/YF0t7pdnd48BPD2keUvdbB0j4hgicsWR7e/q/Dt165f+/L46tvH779pbEvN4fit9AC+wAGHH3x2fGn782E1hXwABOqcJxt97HkrcoULvAonTu/jR5n+huWWHFS7ZskUjEbtSHz0iDJycz36cuRlkN2xXHsmg2gAS6W0eU3NttPpFsQ8UX5BXxLLZ9xfBAt9c3jVz6evvr7w+t/OLxxY3rztaO7H3FhTd//+uidF+kbIt9/89r09u+nN/mbcfj2z4+/fXP6yt3pjdeObn1qPHNhFSG+/vLw+pvT1z8//u7W9O6H0xuvcgH+z9UXhHl0kzYeNrmCGMssPA0mXHsJ61+g09AIQLK++bcm3u4Ik745vfHZg6vXcdKXPlDUff/Nrft3Xp++94fptd88uPqL6Z9/wZk4fvEdWPEKK3OogBCKAjoigG5S8EmPP3z56NZb0zsfHb3xCsz14N6vjz99H4SgcBp6xGmgsAARSuLmh0ASBITzJ8WIyVTMLipmD197ZfrpO3pwO735xvTGx9P3Xzy6+fLxd7+5f/cWp2EufhEW/dAZDAgljKNv3pre+EqbBhChXpFXyhjOlxWXlvCtXSDVvDp3r5X+iqfVTuPIt4LnBCDYRTAbo/M1s6IAuPqf/Bo0U3O/WjRTDcMRa4sTIaJwepaheENoJHlSIbY+QKNLBuFLRQxuPLh99fjDn9ckrZEqpFJs7QYmmoOWBd0qcqWHfTj986/QJiLEwaXo8Pq302ufG/uUivHBYNvu3/lY7OhbsGuNo1tfHL7xwf3vfotyvPY5Kvfdm1y+sMlBlVHE125N793l+FETyvaOyB17HqOveIPdd4NwnPIFuBxAjOqX7o7LVDT1Knkop7CchxZWeZ6TRFNaSQ+S8SCUjlNvFhK7FG3sEyHgJ0ri2TQOP/kdWjraByhMIPSggWNeNNqrXdxPxrl+LQRGtzHudzDLwftG2ICh7YGzzwNBiMuTVVO7OAKyoQ55gaQNSfgI9thBNc3A6zdYmpAxNa9LjikuFJS0J3g2DX0N+YZcnMYvB3FBgb89/uq/j78D4/fO9Ks/Hd6+DjZ1evN1Ye9BNbiFBF2afvgCmA+AuH/nKqjY0a07h7fvcj0CyOPPvpq+dO3wd1cP3/2gWYNOvAkxpCVWJQQQI/d1cj8nWeP3Crt4MwnHnhHF/l7t0N7kBn763UsP3rsHa290/3L1TYI17n/97w/e/qIS26u7P/rlISBIq7zJoBYM1mf3pjfeEjvu2vSTt49ffeH7b66bswPa8mSaWlSWVqhFb8baYbDEdfnwV1+ALhv7qEziYseBcfzRyw/+8z+A879c/fCE5SB+cf/pV0Pq6qRuMzRf4agN+LHxLD9lTveM80//45MXjTgK91p07WGbyWwPjCKj64Z/A8tpRGxHtQSQAteQDv0WLyz0VYDVHl3GDZdYScpgW/UfaT+yiNsGGxcfcR6R+w06QKPywA2hCSbpFzpWJz7O2l4YZ7Cl/QaBMfxyIg/SkJhmiOJLT/hGy2vXIYu6L4dS5brigl6rdE2QVzEbv4Ip1aOqpEPz6JfvHl67WVYRVNc7H/01uipOxNRVj5oeSz5pjVEEvSbBhkF0WZMJasoTeTwKPG7VPfqfklBlFlHxL3ZjfYDsID6QK67rhvIJUvD07aDZuwcsHg9y0Q0A7MGcfVI4wJo5VaZUJSHcuYDAp6++O9sasl2PJbnxJH0gx26GbZUirXS2M6YlPt7/4/EXHyATdM0UcNhthxyS4xz0ILvZ9eaxpnkVJcEebqBxRCsl6i5OfLkymOxAWHGRhR44hRbg3xPRPUS2rPM928FfvcfzE4NDLOn9mIb2eCp5Qtb40AnjvFxRc1SCgp5Ql8PbH4v4jSdcFU/6y28hDNbTHMg9SplOfZ4i9uH6d3TvzcPfvoue/aU/NgSJMBfXUEi5pXLLDq5Rxr7UPtnOgwUYwEML0XwLJV4RTHUJy1JqZgBL2GLCXq1KiN9aWwiw6MIVm+pqjoNpuOOY8r4J5uQL/wuTrwiV', '8d129b6afbeb800fc4da3dc7e6d58d6b4bd1c7bbecf409c403cab508ee30c0dd'),
    'extractor.py': ('eNqNV11v1EYUfd9fMU0fbNPFbCpQ1cA+IARvVJFA7UMUWRPvOJlm7LE843w0WglVFIqAkopCUaFCrarSPiCqFpW0QeK/tNlN8sRf6L0ztrMfNsVSsuuZO3fO3Hvm3LszMzOf8KQn14/Df5Yy+JdowjZ0RkPNZUIUy9Z4yE6TjOWKEb3CiMrTVHDWIxfnT5L1jGuW+TMzM60okzEJgijXecaCgPA4lZkmNEmkpuhMtVrFmFTlN83iNOKCVe88ZtZTSvWK4Eulm3l4rdbHIU0DLYM4PUmoIqHMWKvV6rGohB6gT+UqmWchU20Spar7kUxYmwAYHm2aF2+uReDhEU4TrnCO4ARg7hEX37jiidI0CZkLNm3CE+0RmRnLWXKma1bCx4cdeAp/+GSUQ7Q+piJn57NMZq4z2Hmy/9X1wasvDn/YHT54Rmb/uXLXrCJ7f908/Pb5/ndXh/eeD+/95njGSwGddIngSrs9Hmof47LKNpWLwXBTz8+YkmKNuR6JAFQK8Mp1nvXCYq7BhT00AncFjZd6lLA1SPScOa21BFe50LjfwqIFoGmmIctdkxM/luBEJjx0rT1uiJzZaBdb4uYsyWOWUc3KyI+EBDMNEUzzJTjQinG8sNgudzuCAONbjl3uzAGKrPDltYkDmHSuYNiJKCS458CYzHWaaxxcWOxXvvDg7pazChCNOVijsUEMAwXyEYdpJgEvpHsZ7er273uVd0GVDmDFMkDGkHVa1RSSsJxxwf0ycG6JKvi/zvjyih6JSGmepz0MWc/wU8NlERNG+CQyETKkYnzvGrP1poyNPsB53I6csfsZwuPS4xNHO0M6/uypaTQ2XXDRk6mpydiA2ymbqJSXLol54s62SUw33E7bgjpRBMFDmBYfE3CdOlN+JrJc7urYzC7bBJoUTGS+9kAEPDDIdKLBzjWG5D2TPPioIB8rEwk4BUsqpsPEbKfT6JkJmiqGMG2ci+vVH8+NjWnBiCNS6WxzPAcQGJQgu7fPldE710hTMabyKOIbvpDrLIOJd7rE8VE2nelkTovVwbMXh1duDG/+Onj6YPDoF9AmYhaT4f3re7t/OuOgixuIF1fLlIdzJYZ1rleChMYgns5WiQt0oB9sWXh9HxTc8RpiZjTGeASNMfaoMSj2/rmzFy/5UHli5Xr9ycjQZNNNfbYBuqlGpbHA6a/hSWGm5pZZCfKLG2nVoeuoVZ6mRm1iYBfwqesMXvw+fHRjsH1rsH0bC+Hrl7dg6ODFHwevrr9++XBv5/bBz9f2H963Afv3yucTp0RCT28PFMupCLCodO1Bl5nG14JlfkozoKepZ7WXuqpLI57K8tRwi6eSr1AGZeJ/qoDteztPIfujxQv4MFW8alJoI7ngAAJnkYwCmjLFir1anTiUQjCo4FSI4tTtI0EFZu4O7twnF8+dnYd0gGJ0/A9OedPbY9JDisWoTdyCQzb/3nihKklRkMmrj5KBiMJqoC4Yh4u1lorGqWDVaSKeUME/Y65Z2bZ0rid81IPuBCDjNkVX5MereF9SN80Y0L/r+A5IkYVsbhK8OYHTJDrNj71O4A8IpTkVEMsez7pjLKtHKZUfCqngSvfqDRAvHMG0KHicZivl0xRbTtfFtyo/9QtMNE23iV1fsaIIdtvmxedJJMvvcAYaw8wR794qShXTImf/m8fDL7fJViU4Rdr7jmUdxN4yDJQfquT7p9r2o+EAPSZKvLVsHY0BUtREqJ6NcNEhDZYroOtYrxrP9i6xDb6CG2lWAJOgk1fARPiaChpiu0uMWkLzQ6AsgeacfoO/PBF8lZEVmvWgN03g5kKPSYWSZF1m8CbR24Wzl0/AH1mTAm6a8hv9wUEssHES1NrXi+aIJ4TzFn6qHrQkYJNxQznIw9B2GYV4dBewTUxHSk21w+L/sa6sKJEpKXe2Qd0Gd27tP9wZPvr74Mm1wx+/xtJi9JdsHZG5Ty7MXxovK2wjZKkm580HdirwywjGxgOGCI84VsGcjmohMphsiGnMTWscyNXu5Sxnk02LlfriKEbuoeDrzZS5AMDzA9MEBEF/jmzBQN+plht5FJvTGG0SgzdcBBz+X3jFr5oyz/bVa/qVYKdrficcO2an+o1LR1vPmh8ZphV1bDMw+Onq/va1/d27w+8fT0l3XQc629BqjjaVkx3/VIdZdJZFQFr/AYXN1c0=', '6565a4fc10a165302bf4c4b2fd89c1ef9525638208339d67fc71de34af52fc5a'),
    'ros2_video.py': ('eNqtVt9v2zYQfvdfceOATUIVxUnTonOrAUPWYXnoGjRFX7JAYCXK5iyRAknV8YLsb98dKcly4qHZML1IOh4/3o/v7sgY+/D+6hSi858/xHCum9YIa0V50fClgF/T05cv4OhHeHd5loARnZVqCU3B29zpvGnPvre0BBsjnTApY2xWGd1Anled64zIc5BNq40DrpR23Emt7GzWy7QdvqwzXeGGPyeatpK1GP9lIwJuy92qlp8H0Ev8HdEmVgG3UGgjZjPyLf/4/vLiHDJgxyvBy+OCN8LwY0ke5sXoMZvNZqWo8Axjp/LcK0aNXcaLGeCDXlqhrDY5yuzxg5glgJGEWjpXiyOhSskVBTCqtGm4S6Dkjsc+UoQlK0CQ65Mb+CaDk4BPj+HSCvjE6068NUabiBHqZ7kcIJ0s+RpKWXbrTi1Z7HdqdPIMnsEr+PrzLQiFEbNd7bMCFBphcLN1vGk9nEK4kJm0Uy0v1jklIWJvLlhCZieg4+v5zWs891k4WE3xK4OBzmU5mhahHjyP4Tv46/l/OcDvqRqHuyhoekF46iY1iCHb6DP7fc7itBSFLkXEOlcdvUIcZkRb80LgUq03wkRxj6f+X8OMQMYrsi/ZM6+nVSmcKFxe863uXERE3tHpyunWEmmtxkzg260EBE2QSGVhHJfqNdiCKwuoolW9BWQUdGqt9AZPxXKxI6sQThaWfBAuCtQgZd1SDRcgla+OlGo2R4E2pZ1a1BNTt0TK+e38bCemp9DKSdWJUVjIMvFx9KjdycsIQROYx6NGfmhd79a9wQnkg06DwT+sZFNelpH/jKfGeglkGewqft/qPj3MaHvKpjt5XUeOYtJHjSLlxhid//TuKj6MJJZ6LGKutk8HmQJMC6YWzmdeG7mUitcgbp3hhUMs5DD1uM2Ke0Y00lInnh1uFWTDGr7IUuixT6wFAsrXENa2HNs4L/kCGHKUiiT9Q0sVYVNzog+vjeO4526h65rIS7HzREmgNXpJXS/7TSvR+7bhimqTJIGGaPx6SOlH+vk3ZJycMXw84md2kJ9fpeNTKPlkWj6VgJMIFX1TpEfUU1deIJPKoIZpxpHpo+mFD73B/kObSHf/IN+BaMzgSf8wzBDi+vR0cRM/KGw8Acsrb/m21hx5QDkLWEmA3TkXD9QfkrMzYpD4NKY4T7AJ4Su38k8cjweE8bSB+kN74vUFEIhndWcKBKjaPdYFMfpKt4FeKU7xfF1/EdEAjXOOIn/HggJbUH8ftJH/ZFBnUcwqjr20pJrA9tt2joTXN/gbGjKp+B5yH8aH16FW66HSjXSrXOHgiyp218ssXmju83DtyOcpXlD6eU0Xnf2C2e4z3IOn4lZaZ6NHbYh8SrsW0yOiYH/G7Fq2rTe/wQxgrjNGFzTblXxFFY/ZRPc2OPNTFh9qawF2XMHod7zOMeZDESyFo98+dikyTCjns7LXkIm7J/AmmyLg3w9zfB448qiBke6Km87CyZHfMDE12HdNOgwLYAK/GxRj35m0rT7Vo5LFWw7OzMGrihouUnHg/EOi+8LCvkKpxT3DFTVt1pTdNsL6quRtxlJqqH3eaIXaa47ZsF0V1jFeTvKaUSJMthfE3UnapkWtLZKonMy/wBZPczJjt+I98Hdwuv1GpJgMDiYhHKlUlR6+8Tje4MoudntHoy1E4AATfIk9HW0a3MertHIMWxfmDYW1VOt97dlXSNoVBbKTDfo2u6Zq7Ddjrfmro81qgSMpuBEns8P32IHmFfvkB97al5nPH0ZYYHoMx3zeTbHuwwEo3UXgHn65vBpqQtwWonXw1r9kuJOhbPGIhf3pnolY8G7biggV4zT3TSDP7xdwh4L7cFPwLKv3i9ynddLs94uDVvEO6kPcT/1cr7OPptvvmn3Z/g0NWCCl', '76647bfb1a8863c69e2204bdb334c34a7ce3319319d2c759b6ca6a8f31695754'),
    'worker.py': ('eNp1Uk1rg0AQve+vmNvugkiSY8Bjeyy9i8jWjOkWdWVnDAkh/72zapq00AVR38y+9+bD92OIDHShDL4oDKqNoYfR8WfnP8Av0Xf5VZKSJzz3A2Fks8mAOJoUM3Xd+g7r2uYRKXQnNFZyIw5s7f+M63cMtKtP/oBhScUzR9dwiPfkFZg1SCmKDRQzg0meXDyeym0lOiMJ/hYGBN/CT2hXQVGAdhMHDdiRRAc2T2GrOF72CuR07hImFpaHp/yAjCK+hIyI2zlVJO7Zwo7HoBeKdKQJwvHLtinlZpWBmCzkseWmek4v9UKmxezCNkeT3b+0T9buCglKzh70Cs8Njgwv88uHARzBSrXQXDWFKTao9/MYU10ZaGLHEwmmWye2D1owsTVOnMBSCtA9ErljutfqK19GNGjzuh5cLwtw28MVb/qmxpi6nDYqP0z9SEZUM8CBpoi1o8b74tVJdbIfaRR49rJRqalzM1Yby+RoahrRXIe3teobm//lpw==', 'f7123a096a49d2fd8309e94ad8a2d7e7d51fe37bc1a348cf6a761b481cae3305'),
}
for name, (blob, sha) in _FILES.items():
    data = zlib.decompress(base64.b64decode(blob))
    assert hashlib.sha256(data).hexdigest() == sha, f'{name} rusak'
    Path('/content', name).write_bytes(data)
print('Drive terhubung, skrip siap.')

## Langkah 2 — Atur folder
Ubah path sesuai Drive kamu. `FPS = None` artinya otomatis (baca `session.json` di folder tiap file, kalau tidak ada pakai 30).

`RECURSIVE = True` membaca semua subfolder. Hasilnya meniru struktur subfolder sumber di `DST_DIR`, jadi nama file yang sama di subfolder berbeda tidak bentrok.

In [ ]:
SRC_DIR = '/content/drive/MyDrive/mcap_input'    # folder berisi file .mcap
DST_DIR = '/content/drive/MyDrive/mcap_output'   # folder hasil MP4 (dibuat otomatis)
FPS = None                                       # None = otomatis, atau isi angka bulat, mis. 30
RECURSIVE = True                                 # True = ikut baca semua subfolder di dalam SRC_DIR

## Langkah 3 — Cek daftar file (belum memproses apa pun)

In [ ]:
import os, sys, shutil, psutil

src, dst = Path(SRC_DIR), Path(DST_DIR)
assert src.is_dir(), f'Folder sumber tidak ada: {src}'
dst.mkdir(parents=True, exist_ok=True)

it = src.rglob('*') if RECURSIVE else src.iterdir()
files = sorted(p for p in it if p.is_file() and p.suffix.lower() == '.mcap' and not p.name.startswith('.'))

def out_dir_of(p):
    return dst / p.parent.relative_to(src)

def is_done(p):
    # camera_0 selalu dipublikasikan paling akhir, jadi adanya file ini berarti semua hasil file tsb sudah lengkap
    return (out_dir_of(p) / f'{p.stem}_camera_0.mp4').exists()

todo = []
for p in files:
    done = is_done(p)
    print(f"{'SKIP (sudah ada)' if done else 'PROSES':17} {p.relative_to(src)}  {p.stat().st_size/1e6:,.0f} MB")
    if not done:
        todo.append(p)

print(f"\nTotal {len(files)} file | akan diproses {len(todo)}")
n_sess = sum(1 for d in {p.parent for p in files} if (d / 'session.json').exists())
print(f"Folder dengan session.json: {n_sess} dari {len({p.parent for p in files})} (sisanya pakai FPS di atas / 30)")
print(f"RAM bebas: {psutil.virtual_memory().available/1e9:.1f} GB | disk lokal bebas: {shutil.disk_usage('/content').free/1e9:.0f} GB")

## Langkah 4 — Proses satu per satu
Aman dijalankan ulang: file yang hasilnya sudah ada di Drive dilewati. Kalau ada yang gagal, file lain tetap lanjut.

In [ ]:
import subprocess, time

WORK = Path('/content/work')
ok, failed = [], []
t0 = time.time()

for i, p in enumerate(todo, 1):
    rel = p.relative_to(src)
    print(f"[{i}/{len(todo)}] {rel}", flush=True)
    shutil.rmtree(WORK, ignore_errors=True)
    WORK.mkdir(parents=True)
    try:
        local = WORK / p.name
        shutil.copyfile(p, local)
        if (p.parent / 'session.json').exists():
            shutil.copyfile(p.parent / 'session.json', WORK / 'session.json')

        r = subprocess.run([sys.executable, '/content/worker.py', str(local), 'auto' if FPS is None else str(FPS)],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError((r.stdout.strip().splitlines() or [''])[-1] or r.stderr.strip()[-400:])

        out_dir = out_dir_of(p)
        out_dir.mkdir(parents=True, exist_ok=True)
        outs = sorted(WORK.glob(f'{p.stem}_camera_*.mp4'), reverse=True)   # camera_0 terakhir
        if not outs:
            raise RuntimeError('tidak ada MP4 yang dihasilkan')
        for out_local in outs:
            tmp = out_dir / ('.' + out_local.name + '.partial')
            shutil.copyfile(out_local, tmp)
            os.replace(tmp, out_dir / out_local.name)
        ok.append(str(rel))
        print(f"   selesai ({time.time()-t0:.0f} dtk total)", flush=True)
    except Exception as e:
        failed.append((str(rel), str(e)))
        print(f"   GAGAL: {e}", flush=True)
    finally:
        shutil.rmtree(WORK, ignore_errors=True)

print(f"\nBerhasil {len(ok)} | Gagal {len(failed)}")
for n, e in failed:
    print(' -', n, '->', e)

## Opsional — Diagnosa satu file
Menampilkan daftar topik, jumlah pesan, dan format video di dalam satu MCAP. Dipakai kalau sebuah file gagal diproses. Tidak mengubah apa pun.

In [ ]:
# Isi dengan path satu file .mcap yang bermasalah
FILE = '/content/drive/MyDrive/mcap_input/NAMA_FILE.mcap'

import sys, struct, collections
from pathlib import Path
sys.path.insert(0, '/content')
import mcap_to_mp4 as core

FILE = Path(FILE)
schemas, channels = {}, {}
count = collections.Counter(); first = {}; fmts = collections.defaultdict(collections.Counter)
for op, rec in core.iter_records(FILE):
    if op == 0x03:
        sid, o = core.u16(rec, 0); name, o = core.mstr(rec, o); enc, o = core.mstr(rec, o)
        schemas[sid] = (name, enc)
    elif op == 0x04:
        cid, o = core.u16(rec, 0); sid, o = core.u16(rec, o)
        topic, o = core.mstr(rec, o); menc, o = core.mstr(rec, o)
        channels[cid] = (topic, sid, menc)
    elif op == 0x05:
        cid, _ = core.u16(rec, 0)
        count[cid] += 1
        data = rec[22:]
        if cid not in first:
            first[cid] = data[:24].hex()
        if count[cid] <= 50 and channels.get(cid, ('',))[0].startswith('/ego/camera'):
            try:
                payload, fmt = core.compressed_video_payload(data)
                fmts[cid][(fmt, payload is not None)] += 1
            except Exception as e:
                fmts[cid][('ERR ' + type(e).__name__, False)] += 1

print(f'{FILE.name}: {sum(count.values()):,} pesan, {len(channels)} channel\n')
for cid, (topic, sid, menc) in sorted(channels.items()):
    sch = schemas.get(sid, ('-', '-'))
    print(f'{topic:35} pesan={count[cid]:>8,}  skema={sch[0]} ({sch[1]})  encoding={menc}')
    if cid in fmts: print(f'{"":35} format video (50 pertama): {dict(fmts[cid])}')
    print(f'{"":35} awal data: {first.get(cid, "-")}')

## Opsional — Diagnosa gambar ROS2 (file GoPro)
Untuk MCAP bertopik `/head/camera/image_compressed`. Menampilkan format gambar (JPEG/PNG/H.264/H.265), jumlah frame, perkiraan fps, dan ukuran data. Tidak mengubah apa pun.

In [ ]:
# Diagnosa lanjutan: isi topik gambar ROS2 (sensor_msgs/CompressedImage)
import sys, struct, collections
from pathlib import Path
sys.path.insert(0, '/content')
import mcap_to_mp4 as core

# Isi dengan path file GoPro (.mcap) di Drive
FILE = Path('/content/drive/MyDrive/mcap_input/NAMA_FILE.mcap')
TOPIC = '/head/camera/image_compressed'

def parse(msg):                       # CDR little-endian
    o = 4
    sec, nsec = struct.unpack_from('<II', msg, o); o += 8
    n = struct.unpack_from('<I', msg, o)[0]; o += 4 + n
    o = (o + 3) & ~3
    n = struct.unpack_from('<I', msg, o)[0]; o += 4
    fmt = msg[o:o + n].rstrip(b'\0').decode('utf-8', 'replace'); o += n
    o = (o + 3) & ~3
    n = struct.unpack_from('<I', msg, o)[0]; o += 4
    return sec + nsec / 1e9, fmt, msg[o:o + n]

def signature(d):
    if d[:2] == b'\xff\xd8': return 'JPEG'
    if d[:4] == b'\x89PNG': return 'PNG'
    if d[:4] == b'\x00\x00\x00\x01' or d[:3] == b'\x00\x00\x01':
        nal = d[4] if d[:4] == b'\x00\x00\x00\x01' else d[3]
        return f'AnnexB(h264 nal={nal & 0x1f} / h265 nal={(nal >> 1) & 0x3f})'
    return 'lain: ' + d[:8].hex()

channels, want = {}, None
n = 0; fmts = collections.Counter(); sigs = collections.Counter()
sizes = []; stamps = []; samples = []
for op, rec in core.iter_records(FILE):
    if op == 0x04:
        cid, o = core.u16(rec, 0); _, o = core.u16(rec, o); topic, o = core.mstr(rec, o)
        channels[cid] = topic
        if topic == TOPIC: want = cid
    elif op == 0x05 and want is not None and core.u16(rec, 0)[0] == want:
        t, fmt, d = parse(rec[22:])
        n += 1; fmts[fmt] += 1; sigs[signature(d)] += 1
        sizes.append(len(d)); stamps.append(t)
        if len(samples) < 3: samples.append((t, fmt, len(d), d[:16].hex()))

print(f'{TOPIC}: {n:,} frame')
print('format :', dict(fmts))
print('isi    :', dict(sigs))
print('contoh :', samples)
if n > 1:
    dur = stamps[-1] - stamps[0]
    print(f'durasi : {dur:.1f} dtk  -> kira-kira {(n-1)/dur:.2f} fps' if dur > 0 else 'durasi tidak valid')
    print(f'ukuran per frame: min {min(sizes):,}  rata-rata {sum(sizes)//n:,}  maks {max(sizes):,} byte')
    print(f'total data gambar: {sum(sizes)/1e9:.2f} GB')
